# 03 — Expected Points (EP) Model

**Trinity University Football Analytics**

Builds a Generalized Additive Model (GAM) to estimate Expected Points (EP) for each play situation, then computes Expected Points Added (EPA) for every play on both sides of the ball.

**Input:** `all_df` with engineered features from notebook 02

**Output:** `all_df` with `ep` and `epa` columns merged in

---

### Model Inputs
| Feature | Type | Description |
|---|---|---|
| `dn` | Factor | Down (1–4) |
| `dist` | Spline | Distance to gain |
| `yardline_100` | Spline | Yards to end zone |
| `series` | Spline | Drive number in game |
| `goal_to_go` | Factor | Inside 10 yards of end zone |

### Target
`drive_points` — points scored from that drive forward

## 1. Install & Import

In [ ]:
!pip install pygam -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
import io

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pygam import LinearGAM, s, f

# all_df should be in memory from notebooks 01 & 02
print(f"all_df shape: {all_df.shape}")

## 2. Build Model Dataframe

In [ ]:
# Offense and defense plays only; k plays handled separately
model_df = all_df[all_df['odk'].isin(['o', 'd'])].copy()
k_df     = all_df[all_df['odk'] == 'k'].copy()

# Rename for clarity
model_df = model_df.rename(columns={'yards_to_go': 'yardline_100'})

features = ['dn', 'dist', 'yardline_100', 'series', 'goal_to_go']
target   = 'drive_points'

# Enforce numeric types and drop missing
for col in features + [target]:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

model_df = model_df.dropna(subset=features + [target]).reset_index(drop=True)

print(f"Modeling rows: {len(model_df)}")
print(f"Target distribution:")
print(model_df[target].value_counts().sort_index())

## 3. Train / Test Split

In [ ]:
X = model_df[features].astype(float)
y = model_df[target].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

## 4. Fit GAM

- `f()` = factor term for categorical features (down, goal_to_go)
- `s()` = smooth spline for continuous features (distance, yardline, series)
- `gridsearch()` automatically tunes the smoothing penalty (lambda)

In [ ]:
gam = LinearGAM(
    f(0) +                # dn (1-4, categorical)
    s(1, n_splines=10) +  # dist
    s(2, n_splines=10) +  # yardline_100
    s(3, n_splines=8)  +  # series
    f(4)                  # goal_to_go (0/1, categorical)
).gridsearch(X_train, y_train)

y_pred = gam.predict(X_test)

print(f"R²:        {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print()
gam.summary()

## 5. Predict EP for All Plays

In [ ]:
model_df['ep'] = gam.predict(X)
k_df['ep']    = np.nan  # special teams rows get no EP

model_df = pd.concat([model_df, k_df], ignore_index=True)
model_df = model_df.sort_values(['game_id', 'play']).reset_index(drop=True)

print("EP by unit:")
print(model_df.groupby('odk')['ep'].describe().round(3))

## 6. Compute EPA

EPA logic:
- **Scoring play** (end of drive, score_event ≠ 0): `epa = score_event − ep`
- **Non-scoring drive ending** (punt, turnover, end of game): `epa = 0 − ep`
- **Mid-drive play**: `epa = ep_next − ep`

In [ ]:
# End-of-drive flag
model_df['end_of_drive'] = 0

series_mask = model_df['series'] != 0
model_df.loc[series_mask, 'end_of_drive'] = (
    model_df[series_mask]
    .groupby(['game_id', 'series'])['play']
    .transform('max') == model_df.loc[series_mask, 'play']
).astype(int)

# Scoring plays always end drive
model_df.loc[
    (model_df['odk'].isin(['o', 'd'])) & (model_df['score_event'] != 0),
    'end_of_drive'
] = 1

# ep_next: EP of the next play within same unit's possession
model_df['ep_next'] = model_df.groupby(['game_id', 'odk'])['ep'].shift(-1)
model_df['ep_next'] = model_df['ep_next'].fillna(0)

# EPA formula
model_df['epa'] = (
    (model_df['ep_next'] * (1 - model_df['end_of_drive'])) + model_df['score_event']
) - model_df['ep']

# Special teams scoring plays: EPA = score_event directly
k_score_mask = (model_df['odk'] == 'k') & (model_df['score_event'] != 0)
model_df.loc[k_score_mask, 'epa'] = model_df.loc[k_score_mask, 'score_event']

print("EPA by unit:")
print(model_df.groupby('odk')['epa'].describe().round(3))

## 7. Merge EP & EPA Back to all_df

In [ ]:
all_df = all_df.merge(
    model_df[['game_id', 'play', 'ep', 'epa']],
    on=['game_id', 'play'],
    how='left'
)

print("EP/EPA summary by unit:")
print(all_df.groupby('odk')[['ep', 'epa']].mean().round(3))

## 8. Visualizations

### 8a. EP Curve by Field Position (1st & 10)

In [ ]:
yardlines = np.arange(1, 100)
predict_df = pd.DataFrame({
    'dn': 1, 'dist': 10, 'yardline_100': yardlines, 'series': 10, 'goal_to_go': 0
})
predict_df.loc[predict_df['yardline_100'] <= 10, 'goal_to_go'] = 1
predict_df.loc[predict_df['yardline_100'] <= 10, 'dist'] = predict_df.loc[
    predict_df['yardline_100'] <= 10, 'yardline_100'
]
predict_df['ep'] = gam.predict(predict_df[features])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(predict_df['yardline_100'], predict_df['ep'], color='navy', linewidth=2.5, label='EP (1st & 10)')
ax.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='TD value (7)')
ax.axhline(y=3, color='orange', linestyle='--', alpha=0.5, label='FG value (3)')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No score (0)')
ax.set_xlabel('Yards to End Zone', fontsize=13)
ax.set_ylabel('Expected Points (EP)', fontsize=13)
ax.set_title('Expected Points by Field Position (1st & 10)', fontsize=15)
ax.set_xlim(99, 1)
ax.set_ylim(-2, 8)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axvspan(0, 20, alpha=0.05, color='green')
ax.text(10, 7.5, 'Red Zone', ha='center', fontsize=9, color='green')
ax.text(50, 7.5, 'Midfield', ha='center', fontsize=9, color='gray')
ax.text(85, 7.5, 'Own Territory', ha='center', fontsize=9, color='gray')
plt.tight_layout()

buf = io.BytesIO()
plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
buf.seek(0)
display(Image(buf.read()))
plt.close()

### 8b. EPA Distribution by Unit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, unit, color, label in zip(
    axes, ['o', 'd'], ['steelblue', 'firebrick'], ['Offense', 'Defense']
):
    unit_epa = all_df[all_df['odk'] == unit]['epa'].dropna()
    ax.hist(unit_epa, bins=40, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(unit_epa.mean(), color='black', linestyle='--', label=f'Mean: {unit_epa.mean():.3f}')
    ax.set_title(f'{label} EPA Distribution', fontsize=13)
    ax.set_xlabel('EPA', fontsize=11)
    ax.set_ylabel('Play Count', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('EPA Distribution by Unit', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Export

Uncomment to save the final EPA-enriched dataset.

In [ ]:
# all_df.to_excel('/content/drive/MyDrive/TUFB_EPA/TUFB_EPA_Analysis.xlsx', sheet_name='PlayList', index=False)
# print('Exported successfully.')